# Combined Multi-LLM Wikimedia Context Bridge

This notebook combines the final **immigration** and **women** Wikimedia Context Bridge pipelines with the three-model summary evaluation and selection process from `LLM_Summarisation_Clean.ipynb`.

It performs the complete workflow:

1. Defines the current subgroup-specific article banks.
2. Fetches and cleans Wikimedia articles.
3. Generates one candidate summary per article with each of:
   - `qwen2.5:7b-instruct`
   - `llama3.1:8b`
   - `mistral:7b`
4. evaluates every candidate using ROUGE-L Precision, BERTScore Precision, source-to-summary BARTScore, and length suitability;
5. selects the highest-scoring summary for each article, with optional manual overrides;
6. exports full and report-sized summary metric tables
7. builds subgroup-specific summary banks;
8. maps each comment-subgroup instance to its most similar summary using SBERT;
9. constructs context inputs and saves RoBERTa token-length diagnostics.

The notebook is resumable: fetched articles and generated summaries are reused unless the corresponding force flag is enabled.

## 1. Install dependencies

In [1]:
%pip install -q pandas numpy requests tqdm scikit-learn sentence-transformers transformers rouge-score bert-score sentencepiece accelerate pyarrow


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 2. Imports and configuration

In [2]:
import ast
import gc
import hashlib
import json
import re
import subprocess
import time
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List

import numpy as np
import pandas as pd
import requests
import torch
from IPython.display import display
from bert_score import score as bert_score
from rouge_score import rouge_scorer
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.auto import tqdm
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 300)

In [4]:
# -----------------------------------------------------------------------------
# Paths: edit these two input paths only if your processed files are elsewhere.
# -----------------------------------------------------------------------------
PROJECT_ROOT = Path.cwd()

IMMIGRATION_PROCESSED_PATH = Path(
    "/home/shayan/Distributional-Hate-Speech-Prediction/data/processed/immigration_processed.parquet"
)
WOMEN_PROCESSED_PATH = Path(
    "/home/shayan/Distributional-Hate-Speech-Prediction/notebooks/data/women_processed (1).parquet"
)

OUTPUT_ROOT = PROJECT_ROOT / "combined_context_artifacts"
ARTICLE_DIR = OUTPUT_ROOT / "articles"
SUMMARY_DIR = OUTPUT_ROOT / "summaries"
REPORT_DIR = SUMMARY_DIR / "report_tables"

for directory in [OUTPUT_ROOT, ARTICLE_DIR, SUMMARY_DIR, REPORT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# -----------------------------------------------------------------------------
# Summary generation and evaluation.
# -----------------------------------------------------------------------------
OLLAMA_URL = "http://127.0.0.1:11434/api/generate"
SUMMARISATION_MODELS = [
    "qwen2.5:7b-instruct",
    "llama3.1:8b",
    "mistral:7b",
]

SUMMARY_MIN_WORDS = 50
SUMMARY_MAX_WORDS = 90
MAX_SOURCE_CHARS = 12_000  # matches the final Context Bridge notebooks

BARTSCORE_MODEL_NAME = "facebook/bart-large-cnn"
SBERT_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
ROBERTA_TOKENIZER_NAME = "roberta-base"
TOP_K_SUMMARIES = 1

# Reuse existing outputs by default.
FORCE_REFETCH_ARTICLES = False
FORCE_REGENERATE_SUMMARIES = False
FORCE_RECOMPUTE_METRICS = False
FORCE_RECOMPUTE_EMBEDDINGS = False

# The report table will show all three model candidates for these current articles.
# Edit this list if a different representative pool is preferred.
REPORT_ARTICLE_POOL = {
    "immigration": ["Multiculturalism", "Opposition to immigration", "Liberalism", "Conservatism"],
    "women": ["Misogyny", "Violence against women", "Non-binary gender"],
}

print("Output root:", OUTPUT_ROOT.resolve())
print("Immigration input:", IMMIGRATION_PROCESSED_PATH)
print("Women input:", WOMEN_PROCESSED_PATH)

Output root: /home/shayan/Distributional-Hate-Speech-Prediction/notebooks/summarygeneration/combined_context_artifacts
Immigration input: /home/shayan/Distributional-Hate-Speech-Prediction/data/processed/immigration_processed.parquet
Women input: /home/shayan/Distributional-Hate-Speech-Prediction/notebooks/data/women_processed (1).parquet


## 3. Current subgroup-specific Wikimedia article banks

In [5]:
IMMIGRATION_ARTICLE_BANK = {
    "extremely_liberal": [
        "Left-wing politics",
        "Multiculturalism",
        "Free migration",
    ],
    "liberal": [
        "Liberalism",
        "Left-wing politics",
        "Multiculturalism",
    ],
    "slightly_liberal": [
        "Social liberalism",
        "Liberalism",
        "Left-wing politics",
    ],
    "neutral": [
        "Centrism",
        "Political spectrum",
        "Immigration",
    ],
    "no_opinion": [
        "Centrism",
        "Political spectrum",
        "Immigration",
    ],
    "slightly_conservative": [
        "Conservatism",
        "Right-wing politics",
        "Civic nationalism",
    ],
    "conservative": [
        "Conservatism",
        "Right-wing politics",
        "Opposition to immigration",
    ],
    "extremely_conservative": [
        "National conservatism",
        "Nativism (politics)",
        "Opposition to immigration",
    ],
}

WOMEN_ARTICLE_BANK = {
    "men": [
        "Masculinity",
        "Hegemonic masculinity",
        "Male privilege",
        "Gender role",
        "Toxic masculinity",
    ],
    "women": [
        "Misogyny",
        "Sexism",
        "Patriarchy",
        "Violence against women",
        "Gender inequality",
    ],
    "non_binary": [
        "Non-binary gender",
        "Genderqueer",
        "Gender identity",
        "Gender diversity",
        "Third gender",
    ],
}

DISCOURSE_CONFIG = {
    "immigration": {
        "article_bank": IMMIGRATION_ARTICLE_BANK,
        "processed_path": IMMIGRATION_PROCESSED_PATH,
        "allowed_subgroups": list(IMMIGRATION_ARTICLE_BANK),
        "subgroup_header": "ANNOTATOR IDEOLOGY",
        "output_dir": OUTPUT_ROOT / "immigration",
    },
    "women": {
        "article_bank": WOMEN_ARTICLE_BANK,
        "processed_path": WOMEN_PROCESSED_PATH,
        "allowed_subgroups": ["women", "men", "non_binary"],
        "subgroup_header": "ANNOTATOR GENDER",
        "output_dir": OUTPUT_ROOT / "women",
    },
}

for config in DISCOURSE_CONFIG.values():
    config["output_dir"].mkdir(parents=True, exist_ok=True)

with open(OUTPUT_ROOT / "article_banks.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "immigration": IMMIGRATION_ARTICLE_BANK,
            "women": WOMEN_ARTICLE_BANK,
        },
        f,
        indent=2,
        ensure_ascii=False,
    )

print("Unique immigration articles:", len({x for v in IMMIGRATION_ARTICLE_BANK.values() for x in v}))
print("Unique women articles:", len({x for v in WOMEN_ARTICLE_BANK.values() for x in v}))

Unique immigration articles: 14
Unique women articles: 15


## 4. Build the article catalogue

In [6]:
def slugify(value: str) -> str:
    value = re.sub(r"[^A-Za-z0-9]+", "_", value.strip()).strip("_")
    return value.upper()


def build_article_catalogue() -> pd.DataFrame:
    rows = []
    prefixes = {"immigration": "IMM", "women": "WOMEN"}

    for discourse, config in DISCOURSE_CONFIG.items():
        article_bank = config["article_bank"]
        title_to_groups: Dict[str, List[str]] = {}

        for subgroup, titles in article_bank.items():
            for title in titles:
                title_to_groups.setdefault(title, []).append(subgroup)

        titles = sorted(title_to_groups)

        for title in titles:
            rows.append({
                "summary_id": f"WIKI_{prefixes[discourse]}_{slugify(title)}_001",
                "discourse": discourse,
                "identity_groups": title_to_groups[title],
                "topic": title.lower(),
                "requested_title": title,
            })

    catalogue = pd.DataFrame(rows)

    if catalogue["summary_id"].duplicated().any():
        raise ValueError("Duplicate summary IDs were generated.")

    return catalogue


article_catalogue_df = build_article_catalogue()
print("Articles to process:", len(article_catalogue_df))
display(article_catalogue_df)

Articles to process: 29


,summary_id,discourse,identity_groups,topic,requested_title
0,WIKI_IMM_CENTRISM_001,immigration,"[neutral, no_opinion]",centrism,Centrism
1,WIKI_IMM_CIVIC_NATIONALISM_001,immigration,[slightly_conservative],civic nationalism,Civic nationalism
2,WIKI_IMM_CONSERVATISM_001,immigration,"[slightly_conservative, conservative]",conservatism,Conservatism
3,WIKI_IMM_FREE_MIGRATION_001,immigration,[extremely_liberal],free migration,Free migration
4,WIKI_IMM_IMMIGRATION_001,immigration,"[neutral, no_opinion]",immigration,Immigration
5,WIKI_IMM_LEFT_WING_POLITICS_001,immigration,"[extremely_liberal, liberal, slightly_liberal]",left-wing politics,Left-wing politics
6,WIKI_IMM_LIBERALISM_001,immigration,"[liberal, slightly_liberal]",liberalism,Liberalism
7,WIKI_IMM_MULTICULTURALISM_001,immigration,"[extremely_liberal, liberal]",multiculturalism,Multiculturalism
8,WIKI_IMM_NATIONAL_CONSERVATISM_001,immigration,[extremely_conservative],national conservatism,National conservatism
9,WIKI_IMM_NATIVISM_POLITICS_001,immigration,[extremely_conservative],nativism (politics),Nativism (politics)


## 5. Fetch and clean Wikimedia articles

In [7]:
def fetch_wikipedia_full_extract(title: str, language: str = "en") -> Dict[str, Any]:
    url = f"https://{language}.wikipedia.org/w/api.php"
    params = {
        "action": "query",
        "prop": "extracts|info",
        "explaintext": "1",
        "exsectionformat": "plain",
        "inprop": "url",
        "redirects": "1",
        "titles": title,
        "format": "json",
    }

    try:
        response = requests.get(
            url,
            params=params,
            headers={"User-Agent": "distributional-hate-speech-dissertation/1.0"},
            timeout=30,
        )
        response.raise_for_status()
        data = response.json()
        pages = data.get("query", {}).get("pages", {})
        page = next(iter(pages.values()))

        if "missing" in page:
            raise ValueError(f"Wikipedia page not found: {title}")

        return {
            "requested_title": title,
            "resolved_title": page.get("title"),
            "source_article_id": page.get("pageid"),
            "source_url": page.get("fullurl"),
            "raw_text": page.get("extract"),
            "status_code": response.status_code,
            "fetch_error": None,
        }
    except Exception as exc:
        return {
            "requested_title": title,
            "resolved_title": None,
            "source_article_id": None,
            "source_url": None,
            "raw_text": None,
            "status_code": None,
            "fetch_error": str(exc),
        }


def clean_article_text(text: str) -> str:
    if not isinstance(text, str):
        return ""

    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text).strip()

    stop_markers = [
        "\nReferences\n",
        "\nSee also\n",
        "\nExternal links\n",
        "\nFurther reading\n",
        "\nBibliography\n",
        "\nNotes\n",
    ]
    for marker in stop_markers:
        if marker in text:
            text = text.split(marker, 1)[0].strip()
            break

    return text

In [9]:
import random
import time
import requests

ARTICLES_PATH = ARTICLE_DIR / "wiki_articles.csv"

REQUEST_DELAY_SECONDS = 2.0
REQUEST_JITTER_SECONDS = 1.0

BATCH_SIZE = 5
BATCH_BREAK_SECONDS = 20

MAX_FETCH_RETRIES = 6
INITIAL_BACKOFF_SECONDS = 10


# Load any existing results so the process can continue from a partial run.
if ARTICLES_PATH.exists() and not FORCE_REFETCH_ARTICLES:
    cached_articles_df = pd.read_csv(ARTICLES_PATH)
    print("Loaded cached articles:", ARTICLES_PATH)
else:
    cached_articles_df = pd.DataFrame()


def cached_article_is_valid(summary_id):
    """Check whether an article has already been fetched successfully."""

    if cached_articles_df.empty or "summary_id" not in cached_articles_df.columns:
        return False

    matches = cached_articles_df[
        cached_articles_df["summary_id"] == summary_id
    ]

    if matches.empty:
        return False

    row = matches.iloc[0]

    clean_text = str(row.get("clean_text", ""))
    fetch_error = row.get("fetch_error")

    return (
        len(clean_text) >= 500
        and (pd.isna(fetch_error) or str(fetch_error).strip() == "")
    )


def get_retry_after_seconds(error):
    """Read Wikipedia's Retry-After header when available."""

    response = getattr(error, "response", None)

    if response is None:
        return None

    retry_after = response.headers.get("Retry-After")

    if retry_after is None:
        return None

    try:
        return int(retry_after)
    except ValueError:
        return None


# Keep valid cached rows and retry only missing or failed articles.
article_rows = []

if not cached_articles_df.empty:
    article_rows = cached_articles_df.to_dict("records")


completed_requests = 0

for _, catalogue_row in tqdm(
    article_catalogue_df.iterrows(),
    total=len(article_catalogue_df),
    desc="Fetching Wikimedia articles",
):
    summary_id = catalogue_row["summary_id"]
    requested_title = catalogue_row["requested_title"]

    if cached_article_is_valid(summary_id):
        continue

    fetched_row = None
    last_error = None

    for attempt in range(1, MAX_FETCH_RETRIES + 1):
        try:
            fetched = fetch_wikipedia_full_extract(requested_title)

            # Handle functions that return an error instead of raising it.
            returned_error = fetched.get("fetch_error")

            if returned_error:
                raise RuntimeError(returned_error)

            clean_text = clean_article_text(fetched.get("raw_text"))

            if len(clean_text) < 500:
                raise ValueError(
                    f"Fetched article was too short: {len(clean_text)} characters"
                )

            fetched_row = {
                **catalogue_row.to_dict(),
                **fetched,
                "clean_text": clean_text,
                "char_count": len(clean_text),
                "word_count": len(clean_text.split()),
                "source_prompt_text": clean_text[:MAX_SOURCE_CHARS],
                "fetch_error": None,
            }

            break

        except Exception as error:
            last_error = str(error)

            if attempt == MAX_FETCH_RETRIES:
                print(
                    f"\nFailed to fetch '{requested_title}' "
                    f"after {MAX_FETCH_RETRIES} attempts: {last_error}"
                )
                break

            retry_after = get_retry_after_seconds(error)

            exponential_wait = (
                INITIAL_BACKOFF_SECONDS * (2 ** (attempt - 1))
            )

            wait_seconds = max(
                retry_after or 0,
                exponential_wait,
            )

            wait_seconds += random.uniform(0, 2)

            print(
                f"\nFetch failed for '{requested_title}' "
                f"(attempt {attempt}/{MAX_FETCH_RETRIES})."
            )
            print(f"Error: {last_error}")
            print(f"Waiting {wait_seconds:.1f} seconds before retrying...")

            time.sleep(wait_seconds)

    if fetched_row is None:
        fetched_row = {
            **catalogue_row.to_dict(),
            "requested_title": requested_title,
            "resolved_title": None,
            "raw_text": None,
            "clean_text": "",
            "char_count": 0,
            "word_count": 0,
            "source_prompt_text": "",
            "source_url": None,
            "fetch_error": last_error,
        }

    # Replace any older cached version of the same article.
    article_rows = [
        row
        for row in article_rows
        if row.get("summary_id") != summary_id
    ]
    article_rows.append(fetched_row)

    # Save after every article so a failed run can resume.
    wiki_articles_df = pd.DataFrame(article_rows)
    wiki_articles_df.to_csv(ARTICLES_PATH, index=False)

    completed_requests += 1

    # Normal delay between requests.
    time.sleep(
        REQUEST_DELAY_SECONDS
        + random.uniform(0, REQUEST_JITTER_SECONDS)
    )

    # Longer pause after each batch.
    if completed_requests % BATCH_SIZE == 0:
        print(
            f"\nCompleted {completed_requests} new requests. "
            f"Pausing for {BATCH_BREAK_SECONDS} seconds..."
        )
        time.sleep(BATCH_BREAK_SECONDS)


wiki_articles_df = pd.DataFrame(article_rows)

# Reattach list-valued metadata because CSV stores lists as strings.
wiki_articles_df = (
    wiki_articles_df
    .drop(columns=["identity_groups"], errors="ignore")
    .merge(
        article_catalogue_df[
            ["summary_id", "identity_groups"]
        ],
        on="summary_id",
        how="left",
    )
)

if "fetch_error" not in wiki_articles_df.columns:
    wiki_articles_df["fetch_error"] = None

problem_articles = wiki_articles_df[
    wiki_articles_df["clean_text"].isna()
    | (wiki_articles_df["clean_text"].astype(str).str.len() < 500)
    | wiki_articles_df["fetch_error"].notna()
]

if not problem_articles.empty:
    display(
        problem_articles[
            [
                "discourse",
                "requested_title",
                "resolved_title",
                "char_count",
                "fetch_error",
            ]
        ]
    )

    raise ValueError(
        "One or more Wikimedia articles could not be fetched or were too short. "
        "Run the cell again to retry only the failed articles."
    )

wiki_articles_df.to_csv(ARTICLES_PATH, index=False)

print("Article validation passed:", len(wiki_articles_df), "articles")

display(
    wiki_articles_df[
        [
            "summary_id",
            "discourse",
            "requested_title",
            "resolved_title",
            "word_count",
            "source_url",
        ]
    ].sort_values(["discourse", "requested_title"])
)

Loaded cached articles: /home/shayan/Distributional-Hate-Speech-Prediction/notebooks/summarygeneration/combined_context_artifacts/articles/wiki_articles.csv


Fetching Wikimedia articles:   0%|          | 0/29 [00:00<?, ?it/s]


Completed 5 new requests. Pausing for 20 seconds...

Completed 10 new requests. Pausing for 20 seconds...

Completed 15 new requests. Pausing for 20 seconds...
Article validation passed: 29 articles


,summary_id,discourse,requested_title,resolved_title,word_count,source_url
0,WIKI_IMM_CENTRISM_001,immigration,Centrism,Centrism,3323,https://en.wikipedia.org/wiki/Centrism
1,WIKI_IMM_CIVIC_NATIONALISM_001,immigration,Civic nationalism,Civic nationalism,1095,https://en.wikipedia.org/wiki/Civic_nationalism
2,WIKI_IMM_CONSERVATISM_001,immigration,Conservatism,Conservatism,16010,https://en.wikipedia.org/wiki/Conservatism
3,WIKI_IMM_FREE_MIGRATION_001,immigration,Free migration,Free migration,2278,https://en.wikipedia.org/wiki/Free_migration
4,WIKI_IMM_IMMIGRATION_001,immigration,Immigration,Immigration,6226,https://en.wikipedia.org/wiki/Immigration
5,WIKI_IMM_LEFT_WING_POLITICS_001,immigration,Left-wing politics,Left-wing politics,4352,https://en.wikipedia.org/wiki/Left-wing_politics
6,WIKI_IMM_LIBERALISM_001,immigration,Liberalism,Liberalism,10578,https://en.wikipedia.org/wiki/Liberalism
7,WIKI_IMM_MULTICULTURALISM_001,immigration,Multiculturalism,Multiculturalism,13213,https://en.wikipedia.org/wiki/Multiculturalism
8,WIKI_IMM_NATIONAL_CONSERVATISM_001,immigration,National conservatism,National conservatism,1886,https://en.wikipedia.org/wiki/National_conservatism
9,WIKI_IMM_NATIVISM_POLITICS_001,immigration,Nativism (politics),Nativism (politics),5864,https://en.wikipedia.org/wiki/Nativism_(politics)


## 6. Discourse-specific prompts and Ollama helpers

In [10]:
SUMMARY_PROMPT_TEMPLATES = {
    "immigration": """Task:
Write a concise background-knowledge summary grounded only in the provided Wikimedia article text.

Purpose:
The summary will be used as context for a classifier analysing online immigration-related discourse. It should explain relevant social context, political narratives, stereotypes, hostile framings, identity-based prejudice, or discourse patterns that may help interpret how the topic is discussed online.

Rules:
- Do not include any information not supported by the provided article text.
- Do not start the response with \"this is a summary\" or similar phrases.
- Do not classify any specific post.
- Do not say \"this is hate speech\".
- Do not instruct the classifier what label to predict.
- Do not include moral judgement.
- Do not add unsupported claims.
- Avoid generic definitions unless they help explain immigration-related discourse.
- Keep the summary between {min_words} and {max_words} words.
- Write one paragraph only.

Article title:
{article_title}

Article text:
{article_text}

Summary:
""",
    "women": """Task:
Write a concise background-knowledge summary grounded only in the provided Wikimedia article text.

Purpose:
The summary will be used as context for a classifier analysing online discourse targeting women. It should explain relevant social context, gender norms, stereotypes, hostile framings, identity-based prejudice, or discourse patterns that may help interpret how the topic is discussed online.

Rules:
- Do not include any information not supported by the provided article text.
- Do not start the response with \"this is a summary\" or similar phrases.
- Do not classify any specific post.
- Do not say \"this is hate speech\".
- Do not instruct the classifier what label to predict.
- Do not include moral judgement.
- Do not add unsupported claims.
- Avoid generic definitions unless they help explain gender-related discourse.
- Keep the summary between {min_words} and {max_words} words.
- Write one paragraph only.

Article title:
{article_title}

Article text:
{article_text}

Summary:
""",
}


def build_summary_prompt(article_row: pd.Series) -> str:
    template = SUMMARY_PROMPT_TEMPLATES[article_row["discourse"]]
    return template.format(
        min_words=SUMMARY_MIN_WORDS,
        max_words=SUMMARY_MAX_WORDS,
        article_title=article_row["resolved_title"],
        article_text=article_row["source_prompt_text"],
    ).strip()


def ollama_generate(
    prompt: str,
    model_name: str,
    temperature: float = 0.2,
    max_retries: int = 3,
) -> str:
    payload = {
        "model": model_name,
        "prompt": prompt,
        "stream": False,
        "keep_alive": "5m",
        "options": {
            "temperature": temperature,
            "num_predict": 180,
        },
    }

    for attempt in range(1, max_retries + 1):
        try:
            response = requests.post(OLLAMA_URL, json=payload, timeout=240)
            response.raise_for_status()
            return response.json().get("response", "").strip()
        except Exception as exc:
            print(f"Attempt {attempt}/{max_retries} failed for {model_name}: {exc}")
            time.sleep(3)

    return ""


def stop_ollama_model(model_name: str) -> None:
    try:
        result = subprocess.run(
            ["ollama", "stop", model_name],
            capture_output=True,
            text=True,
            check=False,
        )
        message = result.stdout.strip() or result.stderr.strip()
        print(f"Stopped {model_name}: {message}")
    except Exception as exc:
        print(f"Could not stop {model_name}: {exc}")


def clean_summary(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = re.sub(r"^Summary:\s*", "", text.strip(), flags=re.IGNORECASE)
    return re.sub(r"\s+", " ", text).strip()


sample_article = wiki_articles_df.iloc[0]
print(build_summary_prompt(sample_article)[:2_000])

Task:
Write a concise background-knowledge summary grounded only in the provided Wikimedia article text.

Purpose:
The summary will be used as context for a classifier analysing online immigration-related discourse. It should explain relevant social context, political narratives, stereotypes, hostile framings, identity-based prejudice, or discourse patterns that may help interpret how the topic is discussed online.

Rules:
- Do not include any information not supported by the provided article text.
- Do not start the response with "this is a summary" or similar phrases.
- Do not classify any specific post.
- Do not say "this is hate speech".
- Do not instruct the classifier what label to predict.
- Do not include moral judgement.
- Do not add unsupported claims.
- Avoid generic definitions unless they help explain immigration-related discourse.
- Keep the summary between 50 and 90 words.
- Write one paragraph only.

Article title:
Centrism

Article text:
Centrism is the range of politi

## 7. Generate three candidate summaries per article

In [11]:
ALL_SUMMARIES_PATH = SUMMARY_DIR / "all_candidate_summaries.jsonl"

if FORCE_REGENERATE_SUMMARIES and ALL_SUMMARIES_PATH.exists():
    ALL_SUMMARIES_PATH.unlink()
    print("Deleted previous summaries because FORCE_REGENERATE_SUMMARIES=True")

all_summaries: List[Dict[str, Any]] = []
if ALL_SUMMARIES_PATH.exists():
    with open(ALL_SUMMARIES_PATH, "r", encoding="utf-8") as f:
        all_summaries = [json.loads(line) for line in f if line.strip()]

done_keys = {(row["summary_id"], row["model_name"]) for row in all_summaries}
expected_count = len(wiki_articles_df) * len(SUMMARISATION_MODELS)
print(f"Existing candidates: {len(done_keys)}/{expected_count}")

for model_name in SUMMARISATION_MODELS:
    print("=" * 100)
    print("Generating with:", model_name)
    print("=" * 100)

    for _, article_row in tqdm(
        wiki_articles_df.iterrows(),
        total=len(wiki_articles_df),
        desc=model_name,
    ):
        key = (article_row["summary_id"], model_name)
        if key in done_keys:
            continue

        prompt = build_summary_prompt(article_row)
        raw_summary = ollama_generate(prompt, model_name=model_name)
        summary_text = clean_summary(raw_summary)

        record = {
            "summary_id": article_row["summary_id"],
            "discourse": article_row["discourse"],
            "identity_groups": article_row["identity_groups"],
            "topic": article_row["topic"],
            "requested_title": article_row["requested_title"],
            "source_title": article_row["resolved_title"],
            "source_url": article_row["source_url"],
            "source_article_id": str(article_row["source_article_id"]),
            "model_name": model_name,
            "prompt_version": "context_bridge_discourse_specific_v1",
            "summary_text": summary_text,
            "summary_word_count": len(summary_text.split()),
            "source_prompt_char_count": len(article_row["source_prompt_text"]),
            "source_prompt_word_count": len(article_row["source_prompt_text"].split()),
            "max_source_chars": MAX_SOURCE_CHARS,
            "generation_error": "" if summary_text else "empty_generation",
        }

        all_summaries.append(record)
        done_keys.add(key)

        # Save after every candidate so interrupted runs can resume safely.
        with open(ALL_SUMMARIES_PATH, "w", encoding="utf-8") as f:
            for item in all_summaries:
                f.write(json.dumps(item, ensure_ascii=False) + "\n")

    stop_ollama_model(model_name)
    gc.collect()

print(f"Stored candidates: {len(all_summaries)}/{expected_count}")

Existing candidates: 0/87
Generating with: qwen2.5:7b-instruct


qwen2.5:7b-instruct:   0%|          | 0/29 [00:00<?, ?it/s]

Stopped qwen2.5:7b-instruct: 
Generating with: llama3.1:8b


llama3.1:8b:   0%|          | 0/29 [00:00<?, ?it/s]

Stopped llama3.1:8b: 
Generating with: mistral:7b


mistral:7b:   0%|          | 0/29 [00:00<?, ?it/s]

Stopped mistral:7b: 
Stored candidates: 87/87


## 8. Quality checks

In [12]:
summary_df = pd.read_json(ALL_SUMMARIES_PATH, lines=True)

LABEL_LEAKAGE_PHRASES = [
    "this is hate speech",
    "should be classified",
    "classify this",
    "the post is hateful",
    "this comment is hateful",
    "label as",
    "predict",
    "classifier should",
    "the model should",
]


def check_summary_issues(text: str) -> str:
    if not isinstance(text, str) or not text.strip():
        return "empty_summary"

    issues = []
    count = len(text.split())
    lower = text.lower()

    if count < 40:
        issues.append("too_short")
    if count > 100:
        issues.append("too_long")
    if any(phrase in lower for phrase in LABEL_LEAKAGE_PHRASES):
        issues.append("possible_label_leakage")

    return ", ".join(issues)


summary_df["auto_issues"] = summary_df["summary_text"].apply(check_summary_issues)
summary_df.to_csv(SUMMARY_DIR / "candidate_summaries_checked.csv", index=False)

expected_pairs = {
    (sid, model)
    for sid in wiki_articles_df["summary_id"]
    for model in SUMMARISATION_MODELS
}
actual_pairs = set(zip(summary_df["summary_id"], summary_df["model_name"]))
missing_pairs = expected_pairs - actual_pairs

if missing_pairs:
    raise ValueError(f"Missing {len(missing_pairs)} article-model candidates. Example: {next(iter(missing_pairs))}")

if (summary_df["summary_text"].astype(str).str.strip() == "").any():
    display(summary_df[summary_df["summary_text"].astype(str).str.strip() == ""])
    raise ValueError("One or more generated summaries are empty.")

print("Candidate validation passed:", len(summary_df), "summaries")
display(summary_df[["discourse", "source_title", "model_name", "summary_word_count", "auto_issues"]])

Candidate validation passed: 87 summaries


,discourse,source_title,model_name,summary_word_count,auto_issues
0,immigration,Centrism,qwen2.5:7b-instruct,77,
1,immigration,Civic nationalism,qwen2.5:7b-instruct,74,
2,immigration,Conservatism,qwen2.5:7b-instruct,79,
3,immigration,Free migration,qwen2.5:7b-instruct,90,
4,immigration,Immigration,qwen2.5:7b-instruct,82,
...,...,...,...,...,...
82,women,Patriarchy,mistral:7b,128,too_long
83,women,Sexism,mistral:7b,119,too_long
84,women,Third gender,mistral:7b,122,too_long
85,women,Toxic masculinity,mistral:7b,113,too_long


## 9. Build the evaluation dataframe

In [14]:
source_eval_df = (
    wiki_articles_df[
        [
            "summary_id",
            "resolved_title",
            "source_article_id",
            "source_url",
            "source_prompt_text",
        ]
    ]
    .drop_duplicates(subset=["summary_id"])
    .rename(
        columns={
            "resolved_title": "matched_source_title",
            "source_article_id": "matched_source_article_id",
            "source_url": "matched_source_url",
        }
    )
)

eval_df = summary_df.merge(
    source_eval_df,
    on="summary_id",
    how="left",
    validate="many_to_one",
)


# -------------------------------------------------------------------------
# Check that every generated summary has matching source text.
# -------------------------------------------------------------------------
missing_sources = eval_df[
    eval_df["source_prompt_text"].isna()
    | (eval_df["source_prompt_text"].astype(str).str.strip() == "")
]

if not missing_sources.empty:
    display(
        missing_sources[
            [
                "summary_id",
                "source_title",
                "model_name",
            ]
        ]
    )

    raise ValueError(
        "Some summaries could not be matched to their source article text."
    )


# -------------------------------------------------------------------------
# Normalisation functions.
# -------------------------------------------------------------------------
def normalise_text(value):
    if pd.isna(value):
        return ""

    return re.sub(
        r"\s+",
        " ",
        str(value).strip(),
    ).casefold()


def normalise_url(value):
    if pd.isna(value):
        return ""

    return (
        str(value)
        .strip()
        .rstrip("/")
        .replace("http://", "https://")
    )


def normalise_article_id(value):
    """
    Converts equivalent page IDs such as:
        123456
        123456.0
        "123456"
        "123456.0"
    into the same string: "123456".
    """

    if pd.isna(value):
        return ""

    value = str(value).strip()

    try:
        return str(int(float(value)))
    except (TypeError, ValueError):
        return value


# -------------------------------------------------------------------------
# Compare normalised metadata.
# -------------------------------------------------------------------------
eval_df["title_matches"] = (
    eval_df["source_title"].apply(normalise_text)
    == eval_df["matched_source_title"].apply(normalise_text)
)

eval_df["url_matches"] = (
    eval_df["source_url"].apply(normalise_url)
    == eval_df["matched_source_url"].apply(normalise_url)
)

eval_df["article_id_matches"] = (
    eval_df["source_article_id"].apply(normalise_article_id)
    == eval_df["matched_source_article_id"].apply(normalise_article_id)
)


metadata_mismatches = eval_df[
    ~(
        eval_df["title_matches"]
        & eval_df["url_matches"]
        & eval_df["article_id_matches"]
    )
]


if not metadata_mismatches.empty:
    display(
        metadata_mismatches[
            [
                "summary_id",
                "model_name",
                "source_title",
                "matched_source_title",
                "title_matches",
                "source_article_id",
                "matched_source_article_id",
                "article_id_matches",
                "source_url",
                "matched_source_url",
                "url_matches",
            ]
        ]
    )

    raise ValueError(
        "A genuine source metadata mismatch was detected. "
        "The displayed columns show which field differs."
    )


print(
    "All candidates matched to the exact source article used "
    "during summary generation."
)

All candidates matched to the exact source article used during summary generation.


## 10. ROUGE-L Precision

In [15]:
METRICS_PATH = SUMMARY_DIR / "all_summary_metrics.csv"
ROUGE_PATH = SUMMARY_DIR / "rouge_metrics.csv"

if ROUGE_PATH.exists() and not FORCE_RECOMPUTE_METRICS:
    rouge_df = pd.read_csv(ROUGE_PATH)
    print("Loaded cached ROUGE metrics:", ROUGE_PATH)
else:
    rouge_scorer_obj = rouge_scorer.RougeScorer(
        ["rouge1", "rouge2", "rougeL"],
        use_stemmer=True,
    )

    rouge_rows = []
    for _, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc="ROUGE"):
        scores = rouge_scorer_obj.score(row["source_prompt_text"], row["summary_text"])
        rouge_rows.append({
            "summary_id": row["summary_id"],
            "model_name": row["model_name"],
            "rouge1_precision": scores["rouge1"].precision,
            "rouge1_recall": scores["rouge1"].recall,
            "rouge1_f1": scores["rouge1"].fmeasure,
            "rouge2_precision": scores["rouge2"].precision,
            "rouge2_recall": scores["rouge2"].recall,
            "rouge2_f1": scores["rouge2"].fmeasure,
            "rougeL_precision": scores["rougeL"].precision,
            "rougeL_recall": scores["rougeL"].recall,
            "rougeL_f1": scores["rougeL"].fmeasure,
        })

    rouge_df = pd.DataFrame(rouge_rows)
    rouge_df.to_csv(ROUGE_PATH, index=False)
    print("Saved:", ROUGE_PATH)

display(rouge_df.head())

ROUGE:   0%|          | 0/87 [00:00<?, ?it/s]

Saved: /home/shayan/Distributional-Hate-Speech-Prediction/notebooks/summarygeneration/combined_context_artifacts/summaries/rouge_metrics.csv


,summary_id,model_name,rouge1_precision,rouge1_recall,rouge1_f1,rouge2_precision,rouge2_recall,rouge2_f1,rougeL_precision,rougeL_recall,rougeL_f1
0,WIKI_IMM_CENTRISM_001,qwen2.5:7b-instruct,0.941176,0.044199,0.084433,0.452381,0.021006,0.040148,0.694118,0.032597,0.062269
1,WIKI_IMM_CIVIC_NATIONALISM_001,qwen2.5:7b-instruct,0.893333,0.060036,0.112510,0.378378,0.025112,0.047098,0.560000,0.037634,0.070529
2,WIKI_IMM_CONSERVATISM_001,qwen2.5:7b-instruct,0.725000,0.032294,0.061834,0.227848,0.010028,0.019210,0.437500,0.019488,0.037313
3,WIKI_IMM_FREE_MIGRATION_001,qwen2.5:7b-instruct,0.846154,0.040655,0.077582,0.277778,0.013207,0.025214,0.560440,0.026927,0.051385
4,WIKI_IMM_IMMIGRATION_001,qwen2.5:7b-instruct,0.914634,0.040672,0.077882,0.419753,0.018448,0.035343,0.634146,0.028200,0.053998


## 11. BERTScore Precision

In [16]:
BERTSCORE_PATH = SUMMARY_DIR / "bertscore_metrics.csv"

if BERTSCORE_PATH.exists() and not FORCE_RECOMPUTE_METRICS:
    bertscore_df = pd.read_csv(BERTSCORE_PATH)
    print("Loaded cached BERTScore metrics:", BERTSCORE_PATH)
else:
    candidates = eval_df["summary_text"].tolist()
    references = eval_df["source_prompt_text"].tolist()

    P, R, F1 = bert_score(
        candidates,
        references,
        lang="en",
        verbose=True,
        rescale_with_baseline=True,
    )

    bertscore_df = eval_df[["summary_id", "model_name"]].copy()
    bertscore_df["bertscore_precision"] = P.tolist()
    bertscore_df["bertscore_recall"] = R.tolist()
    bertscore_df["bertscore_f1"] = F1.tolist()
    bertscore_df.to_csv(BERTSCORE_PATH, index=False)
    print("Saved:", BERTSCORE_PATH)

display(bertscore_df.head())

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


calculating scores...
computing bert embedding.


  0%|          | 0/2 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/2 [00:00<?, ?it/s]

done in 4.72 seconds, 18.43 sentences/sec
Saved: /home/shayan/Distributional-Hate-Speech-Prediction/notebooks/summarygeneration/combined_context_artifacts/summaries/bertscore_metrics.csv


,summary_id,model_name,bertscore_precision,bertscore_recall,bertscore_f1
0,WIKI_IMM_CENTRISM_001,qwen2.5:7b-instruct,0.357567,0.013835,0.181257
1,WIKI_IMM_CIVIC_NATIONALISM_001,qwen2.5:7b-instruct,0.325718,-0.030434,0.142789
2,WIKI_IMM_CONSERVATISM_001,qwen2.5:7b-instruct,0.203969,-0.127284,0.034390
3,WIKI_IMM_FREE_MIGRATION_001,qwen2.5:7b-instruct,0.156288,-0.214304,-0.034333
4,WIKI_IMM_IMMIGRATION_001,qwen2.5:7b-instruct,0.334375,-0.039562,0.141916


## 12. Source-to-summary BARTScore

In [17]:
BARTSCORE_PATH = SUMMARY_DIR / "bartscore_metrics.csv"


def bartscore_source_to_summary(
    source_text: str,
    summary_text: str,
    max_source_tokens: int = 1024,
    max_summary_tokens: int = 256,
) -> float:
    if not isinstance(source_text, str) or not source_text.strip():
        return np.nan
    if not isinstance(summary_text, str) or not summary_text.strip():
        return np.nan

    source_inputs = bart_tokenizer(
        source_text,
        return_tensors="pt",
        truncation=True,
        max_length=max_source_tokens,
    ).to(device)

    labels = bart_tokenizer(
        summary_text,
        return_tensors="pt",
        truncation=True,
        max_length=max_summary_tokens,
    ).input_ids.to(device)

    with torch.no_grad():
        outputs = bart_model(
            input_ids=source_inputs["input_ids"],
            attention_mask=source_inputs["attention_mask"],
            labels=labels,
        )

    return -outputs.loss.item()  # higher / less negative is better


if BARTSCORE_PATH.exists() and not FORCE_RECOMPUTE_METRICS:
    bartscore_df = pd.read_csv(BARTSCORE_PATH)
    print("Loaded cached BARTScore metrics:", BARTSCORE_PATH)
else:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("BARTScore device:", device)

    bart_tokenizer = AutoTokenizer.from_pretrained(BARTSCORE_MODEL_NAME)
    bart_model = AutoModelForSeq2SeqLM.from_pretrained(BARTSCORE_MODEL_NAME).to(device)
    bart_model.eval()

    bartscore_rows = []
    for _, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc="BARTScore"):
        bartscore_rows.append({
            "summary_id": row["summary_id"],
            "model_name": row["model_name"],
            "bartscore_src_to_sum": bartscore_source_to_summary(
                row["source_prompt_text"],
                row["summary_text"],
            ),
        })

    bartscore_df = pd.DataFrame(bartscore_rows)
    bartscore_df.to_csv(BARTSCORE_PATH, index=False)
    print("Saved:", BARTSCORE_PATH)

display(bartscore_df.head())

BARTScore device: cuda


[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

BARTScore:   0%|          | 0/87 [00:00<?, ?it/s]

Saved: /home/shayan/Distributional-Hate-Speech-Prediction/notebooks/summarygeneration/combined_context_artifacts/summaries/bartscore_metrics.csv


,summary_id,model_name,bartscore_src_to_sum
0,WIKI_IMM_CENTRISM_001,qwen2.5:7b-instruct,-2.447994
1,WIKI_IMM_CIVIC_NATIONALISM_001,qwen2.5:7b-instruct,-2.749238
2,WIKI_IMM_CONSERVATISM_001,qwen2.5:7b-instruct,-3.915574
3,WIKI_IMM_FREE_MIGRATION_001,qwen2.5:7b-instruct,-3.471235
4,WIKI_IMM_IMMIGRATION_001,qwen2.5:7b-instruct,-2.688770


## 13. Combine metrics and select the best candidate per article

In [18]:
def length_suitability(
    word_count: int,
    min_words: int = SUMMARY_MIN_WORDS,
    max_words: int = SUMMARY_MAX_WORDS,
) -> float:
    if min_words <= word_count <= max_words:
        return 1.0
    if word_count < min_words:
        return max(0.0, word_count / min_words)
    return max(0.0, 1.0 - ((word_count - max_words) / max_words))


def minmax_normalize(series: pd.Series) -> pd.Series:
    numeric = pd.to_numeric(series, errors="coerce")
    minimum = numeric.min()
    maximum = numeric.max()
    if pd.isna(minimum) or pd.isna(maximum) or maximum == minimum:
        return pd.Series(np.zeros(len(series)), index=series.index)
    return (numeric - minimum) / (maximum - minimum)


metrics_df = eval_df[
    [
        "summary_id",
        "discourse",
        "identity_groups",
        "topic",
        "requested_title",
        "source_title",
        "source_url",
        "source_article_id",
        "model_name",
        "summary_word_count",
        "summary_text",
        "auto_issues",
    ]
].copy()

metrics_df = metrics_df.merge(rouge_df, on=["summary_id", "model_name"], how="left")
metrics_df = metrics_df.merge(bertscore_df, on=["summary_id", "model_name"], how="left")
metrics_df = metrics_df.merge(bartscore_df, on=["summary_id", "model_name"], how="left")

metrics_df["length_suitability"] = metrics_df["summary_word_count"].apply(length_suitability)
metrics_df["rougeL_precision_norm"] = minmax_normalize(metrics_df["rougeL_precision"])
metrics_df["bertscore_precision_norm"] = minmax_normalize(metrics_df["bertscore_precision"])
metrics_df["bartscore_src_to_sum_norm"] = minmax_normalize(metrics_df["bartscore_src_to_sum"])

# Same weighted selection criterion used in LLM_Summarisation_Clean.ipynb.
metrics_df["composite_score"] = (
    0.25 * metrics_df["rougeL_precision_norm"]
    + 0.25 * metrics_df["bertscore_precision_norm"]
    + 0.30 * metrics_df["bartscore_src_to_sum_norm"]
    + 0.20 * metrics_df["length_suitability"]
)

metrics_df = metrics_df.sort_values(
    ["summary_id", "composite_score"],
    ascending=[True, False],
).reset_index(drop=True)
metrics_df["automatic_rank"] = metrics_df.groupby("summary_id")["composite_score"].rank(
    method="first",
    ascending=False,
).astype(int)
metrics_df["automatic_selected"] = metrics_df["automatic_rank"].eq(1)

metrics_df.to_csv(METRICS_PATH, index=False)
print("Saved:", METRICS_PATH)

display(
    metrics_df[
        [
            "summary_id",
            "topic",
            "source_title",
            "model_name",
            "summary_word_count",
            "rougeL_precision",
            "bertscore_precision",
            "bartscore_src_to_sum",
            "length_suitability",
            "composite_score",
            "automatic_selected",
            "auto_issues",
        ]
    ]
)

Saved: /home/shayan/Distributional-Hate-Speech-Prediction/notebooks/summarygeneration/combined_context_artifacts/summaries/all_summary_metrics.csv


,summary_id,topic,source_title,model_name,summary_word_count,rougeL_precision,bertscore_precision,bartscore_src_to_sum,length_suitability,composite_score,automatic_selected,auto_issues
0,WIKI_IMM_CENTRISM_001,centrism,Centrism,mistral:7b,118,0.840000,0.463972,-2.030954,0.688889,0.791966,True,too_long
1,WIKI_IMM_CENTRISM_001,centrism,Centrism,qwen2.5:7b-instruct,77,0.694118,0.357567,-2.447994,1.000000,0.665331,False,
2,WIKI_IMM_CENTRISM_001,centrism,Centrism,llama3.1:8b,76,0.650000,0.280766,-2.594635,1.000000,0.584619,False,
3,WIKI_IMM_CIVIC_NATIONALISM_001,civic nationalism,Civic nationalism,llama3.1:8b,82,0.597561,0.391479,-2.375385,1.000000,0.630903,True,
4,WIKI_IMM_CIVIC_NATIONALISM_001,civic nationalism,Civic nationalism,qwen2.5:7b-instruct,74,0.560000,0.325718,-2.749238,1.000000,0.532693,False,
...,...,...,...,...,...,...,...,...,...,...,...,...
82,WIKI_WOMEN_TOXIC_MASCULINITY_001,toxic masculinity,Toxic masculinity,mistral:7b,113,0.640351,0.392132,-2.549472,0.744444,0.585786,False,too_long
83,WIKI_WOMEN_TOXIC_MASCULINITY_001,toxic masculinity,Toxic masculinity,qwen2.5:7b-instruct,71,0.549296,0.354131,-2.921987,1.000000,0.519295,False,
84,WIKI_WOMEN_VIOLENCE_AGAINST_WOMEN_001,violence against women,Violence against women,llama3.1:8b,72,0.694444,0.522718,-1.732859,1.000000,0.828118,True,
85,WIKI_WOMEN_VIOLENCE_AGAINST_WOMEN_001,violence against women,Violence against women,mistral:7b,122,0.634146,0.310670,-2.123964,0.644444,0.573415,False,too_long


## 14. Manual review and optional overrides

The automatically selected candidates are shown below with all three alternatives. Review them for factual grounding, clarity, relevance, neutrality, and label leakage.

To override an automatic winner, add an entry to `MANUAL_OVERRIDES` using its `summary_id` and the preferred model name. Leave the dictionary empty when no override is required.

In [19]:
manual_review_sheet = metrics_df[
    [
        "summary_id",
        "discourse",
        "topic",
        "source_title",
        "model_name",
        "summary_word_count",
        "rougeL_precision",
        "bertscore_precision",
        "bartscore_src_to_sum",
        "length_suitability",
        "composite_score",
        "automatic_selected",
        "auto_issues",
        "summary_text",
    ]
].copy()
manual_review_sheet["manual_approved"] = ""
manual_review_sheet["manual_notes"] = ""
manual_review_sheet.to_csv(SUMMARY_DIR / "manual_review_sheet.csv", index=False)

display(manual_review_sheet)

,summary_id,discourse,topic,source_title,model_name,summary_word_count,rougeL_precision,bertscore_precision,bartscore_src_to_sum,length_suitability,composite_score,automatic_selected,auto_issues,summary_text,manual_approved,manual_notes
0,WIKI_IMM_CENTRISM_001,immigration,centrism,Centrism,mistral:7b,118,0.840000,0.463972,-2.030954,0.688889,0.791966,True,too_long,"Centrism, a moderate political ideology between left-wing and right-wing on the left-right spectrum, is associated with liberalism, radical centrism, and agrarianism. It advocates for gradual change, often through welfare state policies with moderate redistributive emphasis. Centrist parties hol...",,
1,WIKI_IMM_CENTRISM_001,immigration,centrism,Centrism,qwen2.5:7b-instruct,77,0.694118,0.357567,-2.447994,1.000000,0.665331,False,,"Centrism occupies the middle ground between left-wing and right-wing politics, often associated with moderate policies and gradual change. In multi-party systems, centrist parties can form coalition governments but are typically junior partners with limited policy-making power. The popularity of...",,
2,WIKI_IMM_CENTRISM_001,immigration,centrism,Centrism,llama3.1:8b,76,0.650000,0.280766,-2.594635,1.000000,0.584619,False,,"In the context of online immigration-related discourse, it's essential to consider that centrism is often associated with moderate politics, gradual change, and support for a liberal welfare state. Centrists may hold strong beliefs aligned with moderate policies or identify as centrist due to a ...",,
3,WIKI_IMM_CIVIC_NATIONALISM_001,immigration,civic nationalism,Civic nationalism,llama3.1:8b,82,0.597561,0.391479,-2.375385,1.000000,0.630903,True,,"Civic nationalism emphasizes shared citizenship and liberal principles over cultural or ethnic identity. It contrasts with ethnic nationalism, which often involves authoritarian rule and exclusionary practices. Civic nationalists argue that individuals need a national identity to lead meaningful...",,
4,WIKI_IMM_CIVIC_NATIONALISM_001,immigration,civic nationalism,Civic nationalism,qwen2.5:7b-instruct,74,0.560000,0.325718,-2.749238,1.000000,0.532693,False,,"Civic nationalism emphasizes shared citizenship and liberal principles over ethnicity, promoting individual rights and voluntary membership in the nation-state. It contrasts with ethnic nationalism, which is often linked to authoritarian rule. Critics argue that civic nationalism can be as exclu...",,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
82,WIKI_WOMEN_TOXIC_MASCULINITY_001,women,toxic masculinity,Toxic masculinity,mistral:7b,113,0.640351,0.392132,-2.549472,0.744444,0.585786,False,too_long,"The term ""toxic masculinity"" refers to harmful aspects of traditional masculine norms, such as misogyny, homophobia, and violent domination. These traits are associated with negative effects on mental and physical health, including increased psychological problems like depression and substance u...",,
83,WIKI_WOMEN_TOXIC_MASCULINITY_001,women,toxic masculinity,Toxic masculinity,qwen2.5:7b-instruct,71,0.549296,0.354131,-2.921987,1.000000,0.519295,False,,"Toxic masculinity refers to harmful traditional masculine norms such as violence, dominance, and emotional repression, which can lead to psychological problems in men and contribute to issues like domestic violence. Critics argue the term is poorly defined, promotes essentialism, and risks stigm...",,
84,WIKI_WOMEN_VIOLENCE_AGAINST_WOMEN_001,women,violence against women,Violence against women,llama3.1:8b,72,0.694444,0.522718,-1.732859,1.000000,0.828118,True,,"Violence against women is a manifestation of historically unequal power relations between men and women, with acts of violence often used as a mechanism for subjugating women in society or interpersonal relationships. The majority of perpetrators are men, due to societal dynamics where men hold ...",,
85,WIKI_WOMEN_VIOLENCE_AGAINST_WOMEN_001,women,violence against women,Violence against women,mistral:7b,122

In [20]:
# Example:
# MANUAL_OVERRIDES = {
#     "WIKI_IMM_MULTICULTURALISM_001": "llama3.1:8b",
#     "WIKI_WOMEN_MISOGYNY_001": "qwen2.5:7b-instruct",
# }
MANUAL_OVERRIDES: Dict[str, str] = {}

valid_models = set(SUMMARISATION_MODELS)
unknown_models = set(MANUAL_OVERRIDES.values()) - valid_models
if unknown_models:
    raise ValueError(f"Unknown model names in MANUAL_OVERRIDES: {unknown_models}")

selected_rows = []
for summary_id, group in metrics_df.groupby("summary_id", sort=False):
    if summary_id in MANUAL_OVERRIDES:
        preferred_model = MANUAL_OVERRIDES[summary_id]
        selected = group[group["model_name"] == preferred_model]
        if selected.empty:
            raise ValueError(f"Override model {preferred_model} is missing for {summary_id}")
        row = selected.iloc[0].copy()
        row["selection_source"] = "manual_override"
    else:
        row = group.sort_values("composite_score", ascending=False).iloc[0].copy()
        row["selection_source"] = "automatic_composite"
    selected_rows.append(row)

selected_summaries_df = pd.DataFrame(selected_rows).sort_values(
    ["discourse", "topic"]
).reset_index(drop=True)
selected_summaries_df["selected"] = True

SELECTED_SUMMARIES_PATH = SUMMARY_DIR / "selected_summaries.csv"
selected_summaries_df.to_csv(SELECTED_SUMMARIES_PATH, index=False)

print("Saved:", SELECTED_SUMMARIES_PATH)
display(
    selected_summaries_df[
        [
            "summary_id",
            "discourse",
            "source_title",
            "model_name",
            "summary_word_count",
            "composite_score",
            "selection_source",
            "auto_issues",
            "summary_text",
        ]
    ]
)

Saved: /home/shayan/Distributional-Hate-Speech-Prediction/notebooks/summarygeneration/combined_context_artifacts/summaries/selected_summaries.csv


,summary_id,discourse,source_title,model_name,summary_word_count,composite_score,selection_source,auto_issues,summary_text
0,WIKI_IMM_CENTRISM_001,immigration,Centrism,mistral:7b,118,0.791966,automatic_composite,too_long,"Centrism, a moderate political ideology between left-wing and right-wing on the left-right spectrum, is associated with liberalism, radical centrism, and agrarianism. It advocates for gradual change, often through welfare state policies with moderate redistributive emphasis. Centrist parties hol..."
1,WIKI_IMM_CIVIC_NATIONALISM_001,immigration,Civic nationalism,llama3.1:8b,82,0.630903,automatic_composite,,"Civic nationalism emphasizes shared citizenship and liberal principles over cultural or ethnic identity. It contrasts with ethnic nationalism, which often involves authoritarian rule and exclusionary practices. Civic nationalists argue that individuals need a national identity to lead meaningful..."
2,WIKI_IMM_CONSERVATISM_001,immigration,Conservatism,llama3.1:8b,82,0.782536,automatic_composite,,"Conservatism is a cultural, social, and political philosophy that seeks to promote and preserve traditional institutions, customs, and values. It often emphasizes tradition, authority, hierarchy, and property rights, with some variants being more authoritarian or populist. Conservatives may view..."
3,WIKI_IMM_FREE_MIGRATION_001,immigration,Free migration,qwen2.5:7b-instruct,90,0.367537,automatic_composite,,"The article discusses free migration or open immigration as a position advocating few restrictions on people's ability to move freely between countries. It explores spiritual and religious perspectives that view migration as a journey towards enlightenment or divine purification, emphasizing equ..."
4,WIKI_IMM_IMMIGRATION_001,immigration,Immigration,llama3.1:8b,87,0.591768,automatic_composite,,"Immigration is a complex issue influenced by various push and pull factors, including economic, political, social, and environmental influences. Research suggests that migration can be beneficial to both receiving and sending countries, with positive effects on the economy, but also raises conce..."
5,WIKI_IMM_LEFT_WING_POLITICS_001,immigration,Left-wing politics,llama3.1:8b,70,0.715799,automatic_composite,,"Left-wing politics emphasizes social equality and egalitarianism, often opposing social hierarchies and advocating for empowerment and against oppression. Leftist ideologies vary greatly but typically involve a concern for the disadvantaged and a belief in reducing or abolishing unjustified ineq..."
6,WIKI_IMM_LIBERALISM_001,immigration,Liberalism,mistral:7b,120,0.845514,automatic_composite,too_long,"The article on Liberalism outlines a political and moral philosophy that emphasizes individual rights, liberty, consent of the governed, and equality before the law. Originating in the Age of Enlightenment, liberalism sought to replace hereditary privilege, state religion, and absolute monarchy ..."
7,WIKI_IMM_MULTICULTURALISM_001,immigration,Multiculturalism,llama3.1:8b,71,0.580070,automatic_composite,,"Multiculturalism is often associated with Western nation-states, which have historically promoted national identities through education, conscription, and language standardization. This has led to cultural homogenization, particularly in Europe and North America. However, since the 1970s, some c..."
8,WIKI_IMM_NATIONAL_CONSERVATISM_001,immigration,National conservatism,llama3.1:8b,86,0.799557,automatic_composite,,"National conservatism prioritizes national and cultural identity, often based on a theory of the family as a model for the state. It opposes individualism, universal human rights, and internationalism, instead advocating for nationalism, patriotism, and monoculturalism. National conservatives ty..."
9,WIKI_IMM_NATIVISM_POLITICS_001,immigration,Nativism (politics),llama3.1:8b,94,0.717181,automatic_composite,,"Nativism is a political policy promoting the interests of native-born or indigenous people over those of

## 15. Export the dissertation report table

In [21]:
selected_key_set = set(
    zip(selected_summaries_df["summary_id"], selected_summaries_df["model_name"])
)
metrics_df["selected_final"] = [
    (summary_id, model_name) in selected_key_set
    for summary_id, model_name in zip(metrics_df["summary_id"], metrics_df["model_name"])
]

pool_mask = pd.Series(False, index=metrics_df.index)
for discourse, titles in REPORT_ARTICLE_POOL.items():
    pool_mask |= (
        metrics_df["discourse"].eq(discourse)
        & metrics_df["requested_title"].isin(titles)
    )

report_metrics_df = metrics_df.loc[pool_mask].copy()
missing_report_titles = []
for discourse, titles in REPORT_ARTICLE_POOL.items():
    found = set(
        report_metrics_df.loc[
            report_metrics_df["discourse"].eq(discourse),
            "requested_title",
        ]
    )
    missing_report_titles.extend((discourse, title) for title in titles if title not in found)

if missing_report_titles:
    raise ValueError(f"Report pool contains titles not processed: {missing_report_titles}")

report_metrics_df = report_metrics_df.sort_values(
    ["discourse", "requested_title", "composite_score"],
    ascending=[True, True, False],
)

report_export_df = report_metrics_df[
    [
        "summary_id",
        "topic",
        "source_title",
        "model_name",
        "summary_word_count",
        "rougeL_precision",
        "bertscore_precision",
        "bartscore_src_to_sum",
        "length_suitability",
        "composite_score",
        "auto_issues",
        "selected_final",
    ]
].rename(columns={
    "summary_id": "Summary ID",
    "topic": "Topic",
    "source_title": "Source Article",
    "model_name": "Model",
    "summary_word_count": "Words",
    "rougeL_precision": "ROUGE-L P",
    "bertscore_precision": "BERTScore P",
    "bartscore_src_to_sum": "BARTScore",
    "length_suitability": "Length",
    "composite_score": "Composite",
    "auto_issues": "Issues",
    "selected_final": "Best",
})

numeric_cols = ["ROUGE-L P", "BERTScore P", "BARTScore", "Length", "Composite"]
report_export_df[numeric_cols] = report_export_df[numeric_cols].round(4)

REPORT_CSV_PATH = REPORT_DIR / "report_summary_metrics.csv"
REPORT_TEX_PATH = REPORT_DIR / "report_summary_metrics.tex"
REPORT_HTML_PATH = REPORT_DIR / "report_summary_metrics.html"

report_export_df.to_csv(REPORT_CSV_PATH, index=False)
report_export_df.to_html(REPORT_HTML_PATH, index=False)

# The LaTeX version follows the compact table shown in the report example.
latex_df = report_export_df.drop(columns=["Best"])
latex_text = latex_df.to_latex(
    index=False,
    escape=True,
    float_format="%.4f",
    caption=(
        "Automatic summary evaluation for a representative subset of the current "
        "Wikimedia article pool."
    ),
    label="tab:summary_evaluation_subset",
)
REPORT_TEX_PATH.write_text(latex_text, encoding="utf-8")

print("Saved:", REPORT_CSV_PATH)
print("Saved:", REPORT_TEX_PATH)
print("Saved:", REPORT_HTML_PATH)
display(report_export_df)

Saved: /home/shayan/Distributional-Hate-Speech-Prediction/notebooks/summarygeneration/combined_context_artifacts/summaries/report_tables/report_summary_metrics.csv
Saved: /home/shayan/Distributional-Hate-Speech-Prediction/notebooks/summarygeneration/combined_context_artifacts/summaries/report_tables/report_summary_metrics.tex
Saved: /home/shayan/Distributional-Hate-Speech-Prediction/notebooks/summarygeneration/combined_context_artifacts/summaries/report_tables/report_summary_metrics.html


,Summary ID,Topic,Source Article,Model,Words,ROUGE-L P,BERTScore P,BARTScore,Length,Composite,Issues,Best
6,WIKI_IMM_CONSERVATISM_001,conservatism,Conservatism,llama3.1:8b,82,0.7317,0.3991,-1.8153,1.0000,0.7825,,True
7,WIKI_IMM_CONSERVATISM_001,conservatism,Conservatism,mistral:7b,113,0.5948,0.4395,-2.1670,0.7444,0.6254,too_long,False
8,WIKI_IMM_CONSERVATISM_001,conservatism,Conservatism,qwen2.5:7b-instruct,79,0.4375,0.2040,-3.9156,1.0000,0.2626,,False
18,WIKI_IMM_LIBERALISM_001,liberalism,Liberalism,mistral:7b,120,0.7603,0.5270,-1.3783,0.6667,0.8455,too_long,True
19,WIKI_IMM_LIBERALISM_001,liberalism,Liberalism,llama3.1:8b,88,0.6705,0.3573,-2.4047,1.0000,0.6558,,False
20,WIKI_IMM_LIBERALISM_001,liberalism,Liberalism,qwen2.5:7b-instruct,74,0.5676,0.2699,-3.2571,1.0000,0.4510,,False
21,WIKI_IMM_MULTICULTURALISM_001,multiculturalism,Multiculturalism,llama3.1:8b,71,0.7361,0.1227,-2.4435,1.0000,0.5801,,True
22,WIKI_IMM_MULTICULTURALISM_001,multiculturalism,Multiculturalism,mistral:7b,130,0.5373,0.3185,-2.6330,0.5556,0.4402,too_long,False
23,WIKI_IMM_MULTICULTURALISM_001,multiculturalism,Multiculturalism,qwen2.5:7b-instruct,101,0.4314,0.1523,-2.8590,0.8778,0.3343,too_long,False
30,WIKI_IMM_OPPOSITION_TO_IMMIGRATION_001,opposition to immigration,Opposition to immigration,llama3.1:8b,80,0.6875,0.2448,-3.6782,1.0000,0.4629,,True


## 16. Export model-level ranking and a full text report

In [22]:
model_ranking_df = (
    metrics_df.groupby("model_name")
    [
        [
            "rougeL_precision",
            "bertscore_precision",
            "bartscore_src_to_sum",
            "length_suitability",
            "composite_score",
            "summary_word_count",
        ]
    ]
    .mean()
    .round(4)
    .sort_values("composite_score", ascending=False)
)
model_ranking_df.to_csv(SUMMARY_DIR / "model_level_ranking.csv")

def compact_text_table(df: pd.DataFrame, width: int = 42) -> str:
    copy = df.copy()
    for column in copy.columns:
        if pd.api.types.is_numeric_dtype(copy[column]):
            copy[column] = copy[column].round(4)
        copy[column] = copy[column].astype(str).apply(
            lambda value: value if len(value) <= width else value[: width - 3] + "..."
        )
    return copy.to_string(index=False)

lines = [
    "MULTI-LLM WIKIMEDIA SUMMARY SELECTION REPORT",
    f"Generated: {datetime.now():%Y-%m-%d %H:%M:%S}",
    "",
    "MODEL-LEVEL AVERAGE SCORES",
    compact_text_table(model_ranking_df.reset_index()),
    "",
    "SELECTED SUMMARY PER ARTICLE",
    compact_text_table(
        selected_summaries_df[
            [
                "summary_id",
                "discourse",
                "source_title",
                "model_name",
                "summary_word_count",
                "rougeL_precision",
                "bertscore_precision",
                "bartscore_src_to_sum",
                "length_suitability",
                "composite_score",
                "selection_source",
                "auto_issues",
            ]
        ]
    ),
]

FULL_REPORT_PATH = SUMMARY_DIR / "summary_selection_report.txt"
FULL_REPORT_PATH.write_text("\n".join(lines), encoding="utf-8")
print("Saved:", FULL_REPORT_PATH)
display(model_ranking_df)

Saved: /home/shayan/Distributional-Hate-Speech-Prediction/notebooks/summarygeneration/combined_context_artifacts/summaries/summary_selection_report.txt


,rougeL_precision,bertscore_precision,bartscore_src_to_sum,length_suitability,composite_score,summary_word_count
model_name,,,,,,
llama3.1:8b,0.6516,0.3614,-2.1206,0.9739,0.6744,85.1724
mistral:7b,0.6387,0.3406,-2.2647,0.6701,0.5790,119.6897
qwen2.5:7b-instruct,0.5796,0.2705,-2.8474,0.9705,0.5011,82.8621


## 17. Build subgroup-specific summary banks

In [23]:
def build_subgroup_summary_bank(
    discourse: str,
    article_bank: Dict[str, List[str]],
    selected_df: pd.DataFrame,
    output_dir: Path,
) -> tuple[Dict[str, List[Dict[str, Any]]], pd.DataFrame]:
    discourse_selected = selected_df[selected_df["discourse"] == discourse].copy()
    title_lookup = {
        row["requested_title"]: row
        for _, row in discourse_selected.iterrows()
    }

    bank: Dict[str, List[Dict[str, Any]]] = {}
    flat_rows = []

    for subgroup, titles in article_bank.items():
        bank[subgroup] = []
        for title in titles:
            if title not in title_lookup:
                raise ValueError(f"No selected summary for {discourse}: {title}")

            row = title_lookup[title]
            item = {
                "subgroup": subgroup,
                "summary_id": row["summary_id"],
                "article_title": row["requested_title"],
                "resolved_title": row["source_title"],
                "page_url": row["source_url"],
                "selected_model": row["model_name"],
                "summary": row["summary_text"],
                "composite_score": float(row["composite_score"]),
                "selection_source": row["selection_source"],
            }
            bank[subgroup].append(item)
            flat_rows.append(item)

    bank_path = output_dir / "subgroup_summary_bank.json"
    flat_path = output_dir / "summary_bank_flat.csv"
    bank_path.write_text(json.dumps(bank, indent=2, ensure_ascii=False), encoding="utf-8")
    flat_df = pd.DataFrame(flat_rows)
    flat_df.to_csv(flat_path, index=False)

    print("Saved:", bank_path)
    print("Saved:", flat_path)
    return bank, flat_df


subgroup_banks = {}
flat_summary_banks = {}
for discourse, config in DISCOURSE_CONFIG.items():
    bank, flat_df = build_subgroup_summary_bank(
        discourse=discourse,
        article_bank=config["article_bank"],
        selected_df=selected_summaries_df,
        output_dir=config["output_dir"],
    )
    subgroup_banks[discourse] = bank
    flat_summary_banks[discourse] = flat_df
    display(flat_df)

Saved: /home/shayan/Distributional-Hate-Speech-Prediction/notebooks/summarygeneration/combined_context_artifacts/immigration/subgroup_summary_bank.json
Saved: /home/shayan/Distributional-Hate-Speech-Prediction/notebooks/summarygeneration/combined_context_artifacts/immigration/summary_bank_flat.csv


,subgroup,summary_id,article_title,resolved_title,page_url,selected_model,summary,composite_score,selection_source
0,extremely_liberal,WIKI_IMM_LEFT_WING_POLITICS_001,Left-wing politics,Left-wing politics,https://en.wikipedia.org/wiki/Left-wing_politics,llama3.1:8b,"Left-wing politics emphasizes social equality and egalitarianism, often opposing social hierarchies and advocating for empowerment and against oppression. Leftist ideologies vary greatly but typically involve a concern for the disadvantaged and a belief in reducing or abolishing unjustified ineq...",0.715799,automatic_composite
1,extremely_liberal,WIKI_IMM_MULTICULTURALISM_001,Multiculturalism,Multiculturalism,https://en.wikipedia.org/wiki/Multiculturalism,llama3.1:8b,"Multiculturalism is often associated with Western nation-states, which have historically promoted national identities through education, conscription, and language standardization. This has led to cultural homogenization, particularly in Europe and North America. However, since the 1970s, some c...",0.580070,automatic_composite
2,extremely_liberal,WIKI_IMM_FREE_MIGRATION_001,Free migration,Free migration,https://en.wikipedia.org/wiki/Free_migration,qwen2.5:7b-instruct,"The article discusses free migration or open immigration as a position advocating few restrictions on people's ability to move freely between countries. It explores spiritual and religious perspectives that view migration as a journey towards enlightenment or divine purification, emphasizing equ...",0.367537,automatic_composite
3,liberal,WIKI_IMM_LIBERALISM_001,Liberalism,Liberalism,https://en.wikipedia.org/wiki/Liberalism,mistral:7b,"The article on Liberalism outlines a political and moral philosophy that emphasizes individual rights, liberty, consent of the governed, and equality before the law. Originating in the Age of Enlightenment, liberalism sought to replace hereditary privilege, state religion, and absolute monarchy ...",0.845514,automatic_composite
4,liberal,WIKI_IMM_LEFT_WING_POLITICS_001,Left-wing politics,Left-wing politics,https://en.wikipedia.org/wiki/Left-wing_politics,llama3.1:8b,"Left-wing politics emphasizes social equality and egalitarianism, often opposing social hierarchies and advocating for empowerment and against oppression. Leftist ideologies vary greatly but typically involve a concern for the disadvantaged and a belief in reducing or abolishing unjustified ineq...",0.715799,automatic_composite
5,liberal,WIKI_IMM_MULTICULTURALISM_001,Multiculturalism,Multiculturalism,https://en.wikipedia.org/wiki/Multiculturalism,llama3.1:8b,"Multiculturalism is often associated with Western nation-states, which have historically promoted national identities through education, conscription, and language standardization. This has led to cultural homogenization, particularly in Europe and North America. However, since the 1970s, some c...",0.580070,automatic_composite
6,slightly_liberal,WIKI_IMM_SOCIAL_LIBERALISM_001,Social liberalism,Social liberalism,https://en.wikipedia.org/wiki/Social_liberalism,llama3.1:8b,"Social liberalism is a political ideology that emphasizes government intervention in addressing social inequalities and ensuring public welfare, while also prioritizing individual rights and autonomy. It combines elements of classical liberalism with a more active role for the state in promoting...",0.757155,automatic_composite
7,slightly_liberal,WIKI_IMM_LIBERALISM_001,Liberalism,Liberalism,https://en.wikipedia.org/wiki/Liberalism,mistral:7b,"The article on Liberalism outlines a political and moral philosophy that emphasizes individual rights, liberty, consent of the governed, and equality before the law. Originating in the Age of Enlightenment, liberalism sought to replace hereditary privilege, state religion, and absolute monarchy ...",0.845514,automatic_composite
8,slightly_liberal,WIKI_IMM_LEFT_WING_POLITICS_001,Left-wing politics,Left-wing politics,https://en.wikipedia.org/wiki/Left-wing_politics,llama3.1:8b,"Left

Saved: /home/shayan/Distributional-Hate-Speech-Prediction/notebooks/summarygeneration/combined_context_artifacts/women/subgroup_summary_bank.json
Saved: /home/shayan/Distributional-Hate-Speech-Prediction/notebooks/summarygeneration/combined_context_artifacts/women/summary_bank_flat.csv


,subgroup,summary_id,article_title,resolved_title,page_url,selected_model,summary,composite_score,selection_source
0,men,WIKI_WOMEN_MASCULINITY_001,Masculinity,Masculinity,https://en.wikipedia.org/wiki/Masculinity,llama3.1:8b,"Masculinity is a socially constructed concept that varies across cultures and historical periods. In Western societies, traditional masculine traits include strength, courage, independence, leadership, dominance, and assertiveness. However, these standards are not universal and can be influenced...",0.789337,automatic_composite
1,men,WIKI_WOMEN_HEGEMONIC_MASCULINITY_001,Hegemonic masculinity,Hegemonic masculinity,https://en.wikipedia.org/wiki/Hegemonic_masculinity,llama3.1:8b,"In Western society, hegemonic masculinity refers to the culturally idealized form of manhood that is socially and hierarchically exclusive. It is characterized by traits such as violence, aggression, stoicism, and competitiveness, which are often internalized by men through socialization and cul...",0.747167,automatic_composite
2,men,WIKI_WOMEN_MALE_PRIVILEGE_001,Male privilege,Male privilege,https://en.wikipedia.org/wiki/Male_privilege,mistral:7b,"Male privilege refers to the systemic advantages and rights granted to men in patriarchal societies, where males hold primary power and predominate in roles of leadership, moral authority, social privilege, and control of property. These privileges can be positive or negative, affecting various ...",0.770936,automatic_composite
3,men,WIKI_WOMEN_GENDER_ROLE_001,Gender role,Gender role,https://en.wikipedia.org/wiki/Gender_role,llama3.1:8b,"Gender roles are social norms that vary across cultures and are often linked to essentialism, the idea that humans have attributes necessary to their identity based on gender. Sociologists distinguish between biological conceptions of sex and sociocultural understandings of gender. Gender roles ...",0.790245,automatic_composite
4,men,WIKI_WOMEN_TOXIC_MASCULINITY_001,Toxic masculinity,Toxic masculinity,https://en.wikipedia.org/wiki/Toxic_masculinity,llama3.1:8b,"Toxic masculinity refers to traditional cultural masculine norms that can be harmful to men, women, and society overall. These norms emphasize dominance, self-reliance, competition, and suppression of emotions, often leading to violence, aggression, and poor mental health in men. The concept is ...",0.699195,automatic_composite
5,women,WIKI_WOMEN_MISOGYNY_001,Misogyny,Misogyny,https://en.wikipedia.org/wiki/Misogyny,llama3.1:8b,"Misogyny is a form of sexism that perpetuates women's subordinate status in patriarchal societies. It can manifest as hatred, contempt, or prejudice against women and girls, often operating through violence, harassment, coercion, and exclusion from full citizenship. Misogyny has been practiced f...",0.746320,automatic_composite
6,women,WIKI_WOMEN_SEXISM_001,Sexism,Sexism,https://en.wikipedia.org/wiki/Sexism,llama3.1:8b,"Sexism is a form of prejudice or discrimination based on one's sex or gender, primarily affecting women and girls. It is linked to gender roles and stereotypes, and can include the belief that one sex is superior to another. Sexism can manifest as hostile sexism, which involves negative attitude...",0.849145,automatic_composite
7,women,WIKI_WOMEN_PATRIARCHY_001,Patriarchy,Patriarchy,https://en.wikipedia.org/wiki/Patriarchy,llama3.1:8b,"Patriarchy is a social system where men hold primary positions of authority, dominating society through control over women. Sociologists attribute this to the process of socialization, which establishes gender roles and norms that serve to maintain power over women. Patriarchal ideology rational...",0.793296,automatic_composite
8,women,WIKI_WOMEN_VIOLENCE_AGAINST_WOMEN_001,Violence against women,Violence against women,https://en.wikipedia.org/wiki/Violence_against_women,llama3.1:8b,"Violence against women is a manifestation of historically unequal power relations between men and women, with acts of violence often used as a mec

## 18. Load processed data and expand comment-subgroup examples

In [24]:
NUM_LABELS = 3


def ensure_dict(value: Any) -> Dict[str, Any]:
    if isinstance(value, dict):
        return value
    if isinstance(value, str):
        return ast.literal_eval(value)
    raise TypeError(f"Expected dict or stringified dict, received {type(value)}")


def is_valid_distribution(
    distribution: Any,
    num_labels: int = NUM_LABELS,
    tolerance: float = 1e-5,
) -> bool:
    if distribution is None:
        return False
    try:
        values = [float(x) for x in distribution]
    except Exception:
        return False
    return (
        len(values) == num_labels
        and all(value >= -tolerance for value in values)
        and abs(sum(values) - 1.0) < tolerance
    )


def expand_subgroup_examples(
    comment_df: pd.DataFrame,
    discourse: str,
    allowed_subgroups: List[str],
    article_bank: Dict[str, List[str]],
) -> pd.DataFrame:
    rows = []
    skipped = {"none": 0, "invalid": 0, "not_allowed": 0}

    for _, row in comment_df.iterrows():
        distributions = ensure_dict(row["subgroup_distributions"])
        counts = ensure_dict(row["subgroup_counts"])

        for subgroup, target_distribution in distributions.items():
            if subgroup not in allowed_subgroups or subgroup not in article_bank:
                skipped["not_allowed"] += 1
                continue
            if target_distribution is None:
                skipped["none"] += 1
                continue
            if not is_valid_distribution(target_distribution):
                skipped["invalid"] += 1
                continue

            target_distribution = [float(x) for x in target_distribution]
            rows.append({
                "experiment": discourse,
                "comment_id": row["comment_id"],
                "split": row["split"],
                "subgroup": subgroup,
                "subgroup_count": int(counts.get(subgroup, 0)),
                "text": row["text"],
                "target_distribution": target_distribution,
                "target_majority_label": int(np.argmax(target_distribution)),
                "target_expected_label": float(
                    np.dot(np.arange(NUM_LABELS), target_distribution)
                ),
            })

    examples = pd.DataFrame(rows)
    print(f"{discourse}: created {len(examples):,} examples; skipped {skipped}")
    return examples


expanded_examples = {}
for discourse, config in DISCOURSE_CONFIG.items():
    processed_path = config["processed_path"]
    if not processed_path.exists():
        raise FileNotFoundError(
            f"Processed {discourse} file not found: {processed_path}\n"
            "Update the path in Section 2 before continuing."
        )

    processed_df = pd.read_parquet(processed_path)
    examples_df = expand_subgroup_examples(
        comment_df=processed_df,
        discourse=discourse,
        allowed_subgroups=config["allowed_subgroups"],
        article_bank=config["article_bank"],
    )
    expanded_examples[discourse] = examples_df

    display(pd.crosstab(examples_df["split"], examples_df["subgroup"]))
    display(examples_df.head())

immigration: created 9,090 examples; skipped {'none': 21302, 'invalid': 0, 'not_allowed': 3799}


subgroup,conservative,extremely_conservative,extremely_liberal,liberal,neutral,no_opinion,slightly_conservative,slightly_liberal
split,,,,,,,,
test,163,46,197,308,257,46,154,213
train,739,252,907,1407,1027,206,756,1066
validation,157,65,181,311,228,37,169,198


,experiment,comment_id,split,subgroup,subgroup_count,text,target_distribution,target_majority_label,target_expected_label
0,immigration,7,test,extremely_liberal,1,\*points to posters asking for palestinian rights and freedom\* theyre antisemetic and want to kill all the jews and dont acknowledge our right to exist ​ cant wait for this prick to die and go straight to hell. ill dig him the hole to get there myself,"[1.0, 0.0, 0.0]",0,0.0
1,immigration,7,test,liberal,1,\*points to posters asking for palestinian rights and freedom\* theyre antisemetic and want to kill all the jews and dont acknowledge our right to exist ​ cant wait for this prick to die and go straight to hell. ill dig him the hole to get there myself,"[0.0, 0.0, 1.0]",2,2.0
2,immigration,10,train,liberal,1,"They'll come back in your plan, also. Plus we pay to jail them for 2 years. With my plan, we jail them after they come back. And deport them again. It really doesn't matter either way until we're build the wall, and shoot the invaders as enemies when they cross the border.","[0.0, 0.0, 1.0]",2,2.0
3,immigration,10,train,neutral,1,"They'll come back in your plan, also. Plus we pay to jail them for 2 years. With my plan, we jail them after they come back. And deport them again. It really doesn't matter either way until we're build the wall, and shoot the invaders as enemies when they cross the border.","[0.0, 0.0, 1.0]",2,2.0
4,immigration,10,train,slightly_conservative,1,"They'll come back in your plan, also. Plus we pay to jail them for 2 years. With my plan, we jail them after they come back. And deport them again. It really doesn't matter either way until we're build the wall, and shoot the invaders as enemies when they cross the border.","[1.0, 0.0, 0.0]",0,0.0


women: created 7,962 examples; skipped {'none': 3897, 'invalid': 0, 'not_allowed': 7906}


subgroup,men,non_binary,women
split,,,
test,576,16,581
train,2723,108,2734
validation,604,16,604


,experiment,comment_id,split,subgroup,subgroup_count,text,target_distribution,target_majority_label,target_expected_label
0,women,6,train,men,1,First off you look cool as fuck! Anyway if we were in the bedroom I think I'd cream your ass then make you suck my cock clean like a whore ;),"[1.0, 0.0, 0.0]",0,0.0
1,women,6,train,women,1,First off you look cool as fuck! Anyway if we were in the bedroom I think I'd cream your ass then make you suck my cock clean like a whore ;),"[1.0, 0.0, 0.0]",0,0.0
2,women,11,validation,men,1,"eat my fuck, bitch","[0.0, 0.0, 1.0]",2,2.0
3,women,11,validation,women,1,"eat my fuck, bitch","[0.0, 1.0, 0.0]",1,1.0
4,women,26,train,men,2,I'd love to rip those panties off and shove my fat cock in her ass.,"[0.5, 0.0, 0.5]",0,1.0


## 19. SBERT embeddings and subgroup-restricted retrieval

In [25]:
sbert = SentenceTransformer(SBERT_MODEL_NAME)


def dataframe_fingerprint(df: pd.DataFrame, columns: List[str]) -> str:
    content = df[columns].astype(str).to_csv(index=False).encode("utf-8")
    return hashlib.sha256(content).hexdigest()


def load_or_encode(
    texts: List[str],
    cache_path: Path,
    metadata_path: Path,
    fingerprint: str,
) -> np.ndarray:
    if (
        cache_path.exists()
        and metadata_path.exists()
        and not FORCE_RECOMPUTE_EMBEDDINGS
    ):
        metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
        if metadata.get("fingerprint") == fingerprint:
            print("Loading cached embeddings:", cache_path)
            return np.load(cache_path)

    embeddings = sbert.encode(
        texts,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    )
    np.save(cache_path, embeddings)
    metadata_path.write_text(
        json.dumps({"fingerprint": fingerprint}, indent=2),
        encoding="utf-8",
    )
    return embeddings


def map_context_for_discourse(
    discourse: str,
    config: Dict[str, Any],
    examples_df: pd.DataFrame,
    summary_bank_df: pd.DataFrame,
) -> pd.DataFrame:
    output_dir = config["output_dir"]

    summary_fingerprint = dataframe_fingerprint(
        summary_bank_df,
        ["subgroup", "summary_id", "summary"],
    )
    summary_embeddings = load_or_encode(
        texts=summary_bank_df["summary"].tolist(),
        cache_path=output_dir / "summary_embeddings.npy",
        metadata_path=output_dir / "summary_embeddings_meta.json",
        fingerprint=summary_fingerprint,
    )

    unique_comments_df = (
        examples_df[["comment_id", "text"]]
        .drop_duplicates("comment_id")
        .reset_index(drop=True)
    )
    unique_comments_df.to_csv(output_dir / "unique_comments_for_context.csv", index=False)

    comment_fingerprint = dataframe_fingerprint(
        unique_comments_df,
        ["comment_id", "text"],
    )
    comment_embeddings = load_or_encode(
        texts=unique_comments_df["text"].tolist(),
        cache_path=output_dir / "comment_embeddings.npy",
        metadata_path=output_dir / "comment_embeddings_meta.json",
        fingerprint=comment_fingerprint,
    )

    comment_id_to_index = {
        comment_id: index
        for index, comment_id in enumerate(unique_comments_df["comment_id"])
    }
    subgroup_to_indices = {
        subgroup: summary_bank_df.index[
            summary_bank_df["subgroup"] == subgroup
        ].tolist()
        for subgroup in config["article_bank"]
    }

    def retrieve(comment_id: Any, subgroup: str) -> List[Dict[str, Any]]:
        comment_index = comment_id_to_index[comment_id]
        comment_embedding = comment_embeddings[comment_index].reshape(1, -1)
        candidate_indices = subgroup_to_indices[subgroup]
        candidate_embeddings = summary_embeddings[candidate_indices]
        similarities = cosine_similarity(comment_embedding, candidate_embeddings)[0]
        ranked_local = np.argsort(-similarities)[:TOP_K_SUMMARIES]

        results = []
        for rank, local_index in enumerate(ranked_local, start=1):
            global_index = candidate_indices[local_index]
            row = summary_bank_df.iloc[global_index]
            results.append({
                "rank": rank,
                "summary_id": row["summary_id"],
                "article_title": row["article_title"],
                "resolved_title": row["resolved_title"],
                "page_url": row["page_url"],
                "selected_model": row["selected_model"],
                "summary": row["summary"],
                "similarity": float(similarities[local_index]),
            })
        return results

    def build_context_input(row: pd.Series, items: List[Dict[str, Any]]) -> str:
        summaries_text = "\n\n".join(
            f"{item['article_title']}:\n{item['summary']}"
            for item in items
        )
        return (
            "### COMMENT TO CLASSIFY\n"
            f"{row['text']}\n\n"
            f"### {config['subgroup_header']}\n"
            f"{row['subgroup']}\n\n"
            "### RETRIEVED BACKGROUND\n"
            f"{summaries_text}"
        )

    mapped_rows = []
    for _, row in tqdm(
        examples_df.iterrows(),
        total=len(examples_df),
        desc=f"Retrieving {discourse} context",
    ):
        items = retrieve(row["comment_id"], row["subgroup"])
        mapped_rows.append({
            **row.to_dict(),
            "retrieved_summary_ids": [item["summary_id"] for item in items],
            "retrieved_article_titles": [item["article_title"] for item in items],
            "retrieved_page_urls": [item["page_url"] for item in items],
            "retrieved_selected_models": [item["selected_model"] for item in items],
            "retrieved_similarities": [item["similarity"] for item in items],
            "retrieved_summaries": [item["summary"] for item in items],
            "context_input_text": build_context_input(row, items),
        })

    return pd.DataFrame(mapped_rows)


context_datasets = {}
for discourse, config in DISCOURSE_CONFIG.items():
    context_datasets[discourse] = map_context_for_discourse(
        discourse=discourse,
        config=config,
        examples_df=expanded_examples[discourse],
        summary_bank_df=flat_summary_banks[discourse],
    )
    print(discourse, context_datasets[discourse].shape)
    display(context_datasets[discourse].head())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/119 [00:00<?, ?it/s]

Retrieving immigration context:   0%|          | 0/9090 [00:00<?, ?it/s]

immigration (9090, 16)


...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/124 [00:00<?, ?it/s]

Retrieving women context:   0%|          | 0/7962 [00:00<?, ?it/s]

women (7962, 16)


,experiment,comment_id,split,subgroup,subgroup_count,text,target_distribution,target_majority_label,target_expected_label,retrieved_summary_ids,retrieved_article_titles,retrieved_page_urls,retrieved_selected_models,retrieved_similarities,retrieved_summaries,context_input_text
0,women,6,train,men,1,First off you look cool as fuck! Anyway if we were in the bedroom I think I'd cream your ass then make you suck my cock clean like a whore ;),"[1.0, 0.0, 0.0]",0,0.0,[WIKI_WOMEN_MALE_PRIVILEGE_001],[Male privilege],[https://en.wikipedia.org/wiki/Male_privilege],[mistral:7b],[0.10473283380270004],"[Male privilege refers to the systemic advantages and rights granted to men in patriarchal societies, where males hold primary power and predominate in roles of leadership, moral authority, social privilege, and control of property. These privileges can be positive or negative, affecting various...",### COMMENT TO CLASSIFY\nFirst off you look cool as fuck! Anyway if we were in the bedroom I think I'd cream your ass then make you suck my cock clean like a whore ;)\n\n### ANNOTATOR GENDER\nmen\n\n### RETRIEVED BACKGROUND\nMale privilege:\nMale privilege refers to the systemic advantages and r...
1,women,6,train,women,1,First off you look cool as fuck! Anyway if we were in the bedroom I think I'd cream your ass then make you suck my cock clean like a whore ;),"[1.0, 0.0, 0.0]",0,0.0,[WIKI_WOMEN_MISOGYNY_001],[Misogyny],[https://en.wikipedia.org/wiki/Misogyny],[llama3.1:8b],[0.08221177756786346],"[Misogyny is a form of sexism that perpetuates women's subordinate status in patriarchal societies. It can manifest as hatred, contempt, or prejudice against women and girls, often operating through violence, harassment, coercion, and exclusion from full citizenship. Misogyny has been practiced ...",### COMMENT TO CLASSIFY\nFirst off you look cool as fuck! Anyway if we were in the bedroom I think I'd cream your ass then make you suck my cock clean like a whore ;)\n\n### ANNOTATOR GENDER\nwomen\n\n### RETRIEVED BACKGROUND\nMisogyny:\nMisogyny is a form of sexism that perpetuates women's subo...
2,women,11,validation,men,1,"eat my fuck, bitch","[0.0, 0.0, 1.0]",2,2.0,[WIKI_WOMEN_HEGEMONIC_MASCULINITY_001],[Hegemonic masculinity],[https://en.wikipedia.org/wiki/Hegemonic_masculinity],[llama3.1:8b],[0.10694806277751923],"[In Western society, hegemonic masculinity refers to the culturally idealized form of manhood that is socially and hierarchically exclusive. It is characterized by traits such as violence, aggression, stoicism, and competitiveness, which are often internalized by men through socialization and cu...","### COMMENT TO CLASSIFY\neat my fuck, bitch\n\n### ANNOTATOR GENDER\nmen\n\n### RETRIEVED BACKGROUND\nHegemonic masculinity:\nIn Western society, hegemonic masculinity refers to the culturally idealized form of manhood that is socially and hierarchically exclusive. It is characterized by traits ..."
3,women,11,validation,women,1,"eat my fuck, bitch","[0.0, 1.0, 0.0]",1,1.0,[WIKI_WOMEN_MISOGYNY_001],[Misogyny],[https://en.wikipedia.org/wiki/Misogyny],[llama3.1:8b],[0.12419144809246063],"[Misogyny is a form of sexism that perpetuates women's subordinate status in patriarchal societies. It can manifest as hatred, contempt, or prejudice against women and girls, often operating through violence, harassment, coercion, and exclusion from full citizenship. Misogyny has been practiced ...","### COMMENT TO CLASSIFY\neat my fuck, bitch\n\n### ANNOTATOR GENDER\nwomen\n\n### RETRIEVED BACKGROUND\nMisogyny:\nMisogyny is a form of sexism that perpetuates women's subordinate status in patriarchal societies. It can manifest as hatred, contempt, or prejudice against women and girls, often o..."
4,women,26,train,men,2,I'd love to rip those panties off and shove my fat cock in her ass.,"[0.5, 0.0, 0.5]",0,1.0,[WIKI_WOMEN_HEGEMONIC_MASCULINITY_001],[Hegemonic masculinity],[https://en.wikipedia.org/wiki/Hegemonic_masculinity],[llama3.1:8b],[0.059146467596292496],"[In

## 20. RoBERTa token-length diagnostics and final exports

In [26]:
roberta_tokenizer = AutoTokenizer.from_pretrained(ROBERTA_TOKENIZER_NAME)


def count_roberta_tokens(text: str) -> int:
    return len(
        roberta_tokenizer(
            text,
            truncation=False,
            add_special_tokens=True,
        )["input_ids"]
    )


length_reports = {}
for discourse, config in DISCOURSE_CONFIG.items():
    context_df = context_datasets[discourse].copy()
    context_df["tweet_token_length"] = context_df["text"].apply(count_roberta_tokens)
    context_df["context_input_token_length"] = context_df["context_input_text"].apply(
        count_roberta_tokens
    )

    report = {
        "discourse": discourse,
        "n_rows": len(context_df),
        "tweet_mean": context_df["tweet_token_length"].mean(),
        "tweet_median": context_df["tweet_token_length"].median(),
        "tweet_p95": context_df["tweet_token_length"].quantile(0.95),
        "tweet_max": context_df["tweet_token_length"].max(),
        "context_mean": context_df["context_input_token_length"].mean(),
        "context_median": context_df["context_input_token_length"].median(),
        "context_p95": context_df["context_input_token_length"].quantile(0.95),
        "context_max": context_df["context_input_token_length"].max(),
        "pct_context_over_256": float(
            (context_df["context_input_token_length"] > 256).mean()
        ),
        "pct_context_over_384": float(
            (context_df["context_input_token_length"] > 384).mean()
        ),
        "pct_context_over_512": float(
            (context_df["context_input_token_length"] > 512).mean()
        ),
    }
    report_df = pd.DataFrame([report])
    length_reports[discourse] = report_df
    context_datasets[discourse] = context_df

    output_dir = config["output_dir"]
    parquet_path = output_dir / f"{discourse}_context_mapped_examples.parquet"
    csv_path = output_dir / f"{discourse}_context_mapped_examples.csv"
    length_path = output_dir / f"{discourse}_context_token_length_report.csv"

    context_df.to_parquet(parquet_path, index=False)
    context_df.to_csv(csv_path, index=False)
    report_df.to_csv(length_path, index=False)

    print("Saved:", parquet_path)
    print("Saved:", csv_path)
    print("Saved:", length_path)
    display(report_df)

combined_length_report_df = pd.concat(length_reports.values(), ignore_index=True)
combined_length_report_df.to_csv(
    OUTPUT_ROOT / "combined_context_token_length_report.csv",
    index=False,
)

Saved: /home/shayan/Distributional-Hate-Speech-Prediction/notebooks/summarygeneration/combined_context_artifacts/immigration/immigration_context_mapped_examples.parquet
Saved: /home/shayan/Distributional-Hate-Speech-Prediction/notebooks/summarygeneration/combined_context_artifacts/immigration/immigration_context_mapped_examples.csv
Saved: /home/shayan/Distributional-Hate-Speech-Prediction/notebooks/summarygeneration/combined_context_artifacts/immigration/immigration_context_token_length_report.csv


,discourse,n_rows,tweet_mean,tweet_median,tweet_p95,tweet_max,context_mean,context_median,context_p95,context_max,pct_context_over_256,pct_context_over_384,pct_context_over_512
0,immigration,9090,37.327833,31.0,88.55,187,182.368757,175.0,249.0,342,0.030693,0.0,0.0


Saved: /home/shayan/Distributional-Hate-Speech-Prediction/notebooks/summarygeneration/combined_context_artifacts/women/women_context_mapped_examples.parquet
Saved: /home/shayan/Distributional-Hate-Speech-Prediction/notebooks/summarygeneration/combined_context_artifacts/women/women_context_mapped_examples.csv
Saved: /home/shayan/Distributional-Hate-Speech-Prediction/notebooks/summarygeneration/combined_context_artifacts/women/women_context_token_length_report.csv


,discourse,n_rows,tweet_mean,tweet_median,tweet_p95,tweet_max,context_mean,context_median,context_p95,context_max,pct_context_over_256,pct_context_over_384,pct_context_over_512
0,women,7962,29.113665,22.0,74.95,143,172.789626,168.0,233.0,341,0.018839,0.0,0.0


## 21. Final inspection checklist

In [ ]:
for discourse, context_df in context_datasets.items():
    print("=" * 100)
    print(discourse.upper())
    print("=" * 100)
    print("Subgroups:", sorted(context_df["subgroup"].unique()))
    print("Retrieved item counts:")
    print(context_df["retrieved_article_titles"].apply(len).value_counts())

    display(
        context_df[
            [
                "comment_id",
                "subgroup",
                "retrieved_article_titles",
                "retrieved_selected_models",
                "retrieved_similarities",
                "retrieved_summaries",
                "context_input_token_length",
                "text",
            ]
        ].head(10)
    )

print("\nAll created files:")
for path in sorted(OUTPUT_ROOT.rglob("*")):
    if path.is_file():
        print("-", path)

### Manual checks before model training

1. Review `summaries/manual_review_sheet.csv` and record any manual overrides.
2. Confirm that `summaries/selected_summaries.csv` contains one selected summary for every current article.
3. Confirm that the report table uses only current articles.
4. Inspect retrieved summaries for several comments under different subgroups.
5. Check that token lengths are suitable for the RoBERTa sequence length used in the model notebooks.
6. Train the context-augmented models using:
   - `combined_context_artifacts/immigration/immigration_context_mapped_examples.parquet`
   - `combined_context_artifacts/women/women_context_mapped_examples.parquet`